### RAG Pipelines- Data Ingestion to Vector Database Pipeline 

In [10]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [12]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data") 

Found 3 PDF files to process.

Processing file: ..\data\pdf\EC4102_MidTerm_Proposal.pdf
Loaded 5 pages

Processing file: ..\data\pdf\Mid_term_applied_econometrics_group_6.pdf
Loaded 5 pages

Processing file: ..\data\pdf\SadakVision — INDIS 2026.pdf
Loaded 16 pages

Total documents loaded: 26


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-29T12:42:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-29T12:42:36+00:00', 'subject': '(unspecified)', 'title': 'Mid-Term Research Proposal - EC4102', 'trapped': '/False', 'source': '..\\data\\pdf\\EC4102_MidTerm_Proposal.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'EC4102_MidTerm_Proposal.pdf', 'file_type': 'pdf'}, page_content="The Teammate and Rival Spillover Effect of a Superstar: Evidence from\n Caitlin Clark and Digital Attention in the WNBA\n Applied Econometrics (EC4102) — Mid-Term Research Proposal\n Group [Group Number] · [Member Names] · Monsoon Semester, AY 2026–27\n1   Motivation\nIn many production settings — firms, research labs, sports teams — a small number of\nexceptionally productive individuals coexist with many ordinary co-workers. A long-standing\nquestion in labor economics is whether such 

### Text Splitting and making Chunks

In [14]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata} ")
    return split_docs
   

In [15]:
chunks = split_documents(all_pdf_documents)

Split 26 documents into 52 chunks

Example chunk:
Content: The Teammate and Rival Spillover Effect of a Superstar: Evidence from
 Caitlin Clark and Digital Attention in the WNBA
 Applied Econometrics (EC4102) — Mid-Term Research Proposal
 Group [Group Number]...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-29T12:42:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-29T12:42:36+00:00', 'subject': '(unspecified)', 'title': 'Mid-Term Research Proposal - EC4102', 'trapped': '/False', 'source': '..\\data\\pdf\\EC4102_MidTerm_Proposal.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'EC4102_MidTerm_Proposal.pdf', 'file_type': 'pdf'} 


### Embedding and Vector Store DB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
    ):
        """Initialize the EmbeddingManager."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) -> None:
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully: {self.model_name}. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [18]:
# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1713.42it/s]


Model loaded successfully: all-MiniLM-L6-v2. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_10348\3387099033.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


### VectorStore

In [19]:
class VectorStore:
    """Handles storage of document embeddings using ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persistent_directory (str): Directory where the vector
                store will be persisted.
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self) -> None:
        """Initialize the persistent ChromaDB vector store."""

        try:
            # Create the directory if it doesn't exist
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ) -> None:
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embeddings for the documents.
        """

        if not documents or embeddings.size == 0:
            raise ValueError(
                "Documents and embeddings cannot be empty"
            )

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must match"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to the vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Store original document text
            documents_text.append(
                doc.page_content
            )

            # Convert NumPy array to Python list
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to the vector store."
            )

            print(
                f"Total documents in collection after addition: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

In [20]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [21]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-29T12:42:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-29T12:42:36+00:00', 'subject': '(unspecified)', 'title': 'Mid-Term Research Proposal - EC4102', 'trapped': '/False', 'source': '..\\data\\pdf\\EC4102_MidTerm_Proposal.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'EC4102_MidTerm_Proposal.pdf', 'file_type': 'pdf'}, page_content='The Teammate and Rival Spillover Effect of a Superstar: Evidence from\n Caitlin Clark and Digital Attention in the WNBA\n Applied Econometrics (EC4102) — Mid-Term Research Proposal\n Group [Group Number] · [Member Names] · Monsoon Semester, AY 2026–27\n1   Motivation\nIn many production settings — firms, research labs, sports teams — a small number of\nexceptionally productive individuals coexist with many ordinary co-workers. A long-standing\nquestion in labor economics is whether such 

In [25]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 52 texts...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.91s/it]


Generated embeddings with shape: (52, 384)
Adding 52 documents to the vector store...
Successfully added 52 documents to the vector store.
Total documents in collection after addition: 52
